# ML1 Task 2 — Focused SVR RBF Modeling Notebook

This notebook is a **separate focused workflow** for improving the Developer Salary Prediction model using only **SVR with RBF kernel** as the final modeling algorithm.

Why this notebook exists:

- The broad modeling notebook showed that **SVR RBF was the strongest model family**.
- The best validation model had much worse internal-test RMSE, so we need to investigate stability rather than blindly trusting one validation split.
- Distance/kernel models such as RBF SVR can suffer when the feature matrix has too many sparse binary variables, so this notebook rebuilds **SVR-specific feature engineering** from the raw data.

Main improvements here:

1. Rebuild preprocessing specifically for RBF SVR.
2. Test smaller SVR-oriented feature profiles: counts-only, top-5 technologies, top-10 technologies.
3. Add optional smoothed region target encoding fitted on training data only.
4. Compare raw, winsorized, floor-500, and drop-below-500 training target variants.
5. Test safer calibration options: no correction, train smearing, conservative multipliers, and validation multiplier.
6. Diagnose why validation and internal-test scores differ.
7. Generate multiple meaningful submission files for public leaderboard testing.

The only predictive algorithm used here is **SVR with RBF kernel**, which is within ML1 scope. Feature engineering and target transformations are used as allowed tools.


In [1]:
# ===============================================================
# 1. Imports and configuration
# ===============================================================
from __future__ import annotations

import json
import math
import pickle
import re
import time
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "annual.pay.usd"
ID_COL = "id"

TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
OUTPUT_DIR = Path("svr_rbf_focused_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Use FAST_MODE=True for a first run. Set to False for a wider search after confirming everything works.
FAST_MODE = True

# Internal split proportions. Same logic as before for comparability.
VAL_SIZE = 0.15
INTERNAL_TEST_SIZE = 0.15
SPLIT_REGION_MIN_COUNT = 10

# Salary safety and target cleaning options.
LOW_SALARY_FLOOR = 500.0
PREDICTION_MIN_USD = 1.0
PREDICTION_MAX_USD = 1_000_000.0

# CV is now used only for top candidates to avoid a huge runtime.
N_SPLITS_CV = 5
N_TOP_FOR_CV = 10 if FAST_MODE else 20

print("Configuration loaded.")
print("FAST_MODE:", FAST_MODE)
print("Output directory:", OUTPUT_DIR.resolve())


Configuration loaded.
FAST_MODE: True
Output directory: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_rbf_focused_outputs


## 2. Load raw data and create the same leakage-safe split

We reload `train.csv` and `test.csv`, then split the labeled data into:

- internal train,
- validation,
- internal test.

The split is stratified by grouped region so that the major salary geography pattern is not accidentally concentrated in one split.


In [2]:
# ===============================================================
# 2. Load raw data and create split
# ===============================================================
def normalize_text_value(value):
    # Normalize text values while preserving missing values.
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(normalize_text_value)
    # Normalize common Other categories.
    replacements = {
        "Other:": "Other",
        "Other (please specify):": "Other",
    }
    for col in ["dev.role", "industry", "first.help.source"]:
        if col in df.columns:
            df[col] = df[col].replace(replacements)
    return df


def make_region_strata(region_series: pd.Series, min_count: int = SPLIT_REGION_MIN_COUNT) -> pd.Series:
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def require_columns(df: pd.DataFrame, required: Iterable[str], df_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")


def make_internal_split(df: pd.DataFrame, seed: int = RANDOM_STATE):
    strata_full = make_region_strata(df["region"]) if "region" in df.columns else None
    df_trainval, df_internal_test = train_test_split(
        df,
        test_size=INTERNAL_TEST_SIZE,
        random_state=seed,
        shuffle=True,
        stratify=strata_full,
    )
    strata_trainval = make_region_strata(df_trainval["region"]) if "region" in df_trainval.columns else None
    relative_val_size = VAL_SIZE / (1.0 - INTERNAL_TEST_SIZE)
    df_train, df_val = train_test_split(
        df_trainval,
        test_size=relative_val_size,
        random_state=seed,
        shuffle=True,
        stratify=strata_trainval,
    )
    return df_train.reset_index(drop=False).rename(columns={"index": "original_index"}),            df_val.reset_index(drop=False).rename(columns={"index": "original_index"}),            df_internal_test.reset_index(drop=False).rename(columns={"index": "original_index"})


df_all = pd.read_csv(TRAIN_PATH)
df_kaggle = pd.read_csv(TEST_PATH)
require_columns(df_all, [TARGET, "region"], "train")
require_columns(df_kaggle, [ID_COL], "test")

df_all = normalize_object_columns(df_all)
df_kaggle = normalize_object_columns(df_kaggle)

df_train, df_val, df_internal_test = make_internal_split(df_all, seed=RANDOM_STATE)

print("Raw shapes")
print("Full train:", df_all.shape)
print("Kaggle test:", df_kaggle.shape)
print("\nInternal split")
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Internal test:", df_internal_test.shape)

# Save split indices for reproducibility.
df_train[["original_index"]].to_csv(OUTPUT_DIR / "train_original_indices.csv", index=False)
df_val[["original_index"]].to_csv(OUTPUT_DIR / "val_original_indices.csv", index=False)
df_internal_test[["original_index"]].to_csv(OUTPUT_DIR / "internal_test_original_indices.csv", index=False)


Raw shapes
Full train: (2512, 41)
Kaggle test: (628, 41)

Internal split
Train: (1758, 42)
Validation: (377, 42)
Internal test: (377, 42)


## 3. Diagnose validation vs internal-test instability

Before tuning again, we check whether the validation and internal-test sets have similar salary distributions.

This matters because RMSE is very sensitive to a few extremely high salaries. A split containing one or two huge salaries can look much worse even if the model is generally reasonable.


In [3]:
# ===============================================================
# 3. Target distribution diagnostics
# ===============================================================
def target_distribution_summary(name: str, y: pd.Series) -> Dict[str, Any]:
    y = pd.Series(y).astype(float)
    return {
        "split": name,
        "n": len(y),
        "mean": y.mean(),
        "median": y.median(),
        "min": y.min(),
        "max": y.max(),
        "lt_500": int((y < 500).sum()),
        "gt_100k": int((y > 100_000).sum()),
        "gt_200k": int((y > 200_000).sum()),
        "p01": y.quantile(0.01),
        "p05": y.quantile(0.05),
        "p25": y.quantile(0.25),
        "p75": y.quantile(0.75),
        "p95": y.quantile(0.95),
        "p99": y.quantile(0.99),
    }

split_summary = pd.DataFrame([
    target_distribution_summary("train", df_train[TARGET]),
    target_distribution_summary("validation", df_val[TARGET]),
    target_distribution_summary("internal_test", df_internal_test[TARGET]),
])

display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "split_target_distribution_summary.csv", index=False)

print("Important interpretation:")
print("If internal_test has more extreme high salaries than validation, internal-test RMSE can be much worse even for the same model.")


,split,n,mean,median,min,max,lt_500,gt_100k,gt_200k,p01,p05,p25,p75,p95,p99
0,train,1758,51287.046075,41854.5,11.0,4773360.0,57,143,13,76.26,882.55,16073.5,67898.0,117887.45,178901.9
1,validation,377,46288.106101,38113.0,1.0,384552.0,13,36,3,183.80,756.20,15122.0,65008.0,121496.40,169134.0
2,internal_test,377,45797.676393,39989.0,25.0,911275.0,11,23,2,148.76,809.20,18590.0,60052.0,116545.40,156503.0


Important interpretation:
If internal_test has more extreme high salaries than validation, internal-test RMSE can be much worse even for the same model.


## 4. Target variants

We will not assume one target treatment is best. For each candidate we can train on:

- `raw`: original log salary,
- `winsorized`: train-only 1%/99% winsorized target,
- `floor500`: salaries below 500 are floored to 500 for training,
- `drop_lt500`: rows below 500 are removed from training.

Validation and internal-test targets remain raw.


In [4]:
# ===============================================================
# 4. Target variant helpers
# ===============================================================
def safe_log_salary(y: pd.Series) -> pd.Series:
    return np.log(pd.Series(y).astype(float).clip(lower=1))


def fit_region_winsor_bounds(df_train_like: pd.DataFrame, lower_q: float = 0.01, upper_q: float = 0.99, min_region_n: int = 30):
    y = df_train_like[TARGET].astype(float)
    global_bounds = (float(y.quantile(lower_q)), float(y.quantile(upper_q)))
    region_bounds = {}
    if "region" not in df_train_like.columns:
        return global_bounds, region_bounds
    for region, group in df_train_like.groupby("region", dropna=False):
        vals = group[TARGET].astype(float)
        if len(vals) >= min_region_n:
            bounds = (float(vals.quantile(lower_q)), float(vals.quantile(upper_q)))
        else:
            bounds = global_bounds
        region_bounds[str(region)] = bounds
    return global_bounds, region_bounds


def apply_region_winsor(y: pd.Series, regions: pd.Series, global_bounds, region_bounds) -> pd.Series:
    y_out = pd.Series(y).astype(float).copy()
    regions = pd.Series(regions).astype(str)
    for region in regions.unique():
        lo, hi = region_bounds.get(str(region), global_bounds)
        mask = regions == str(region)
        y_out.loc[mask] = y_out.loc[mask].clip(lower=lo, upper=hi)
    return y_out


def get_training_target_variant(df_train_like: pd.DataFrame, variant: str):
    # Return df_train_subset, y_log, y_raw_for_training_metric, and mask.
    df = df_train_like.copy().reset_index(drop=True)
    y_raw = df[TARGET].astype(float).copy()

    if variant == "raw":
        mask = pd.Series(True, index=df.index)
        y_target = y_raw.copy()
    elif variant == "winsorized":
        global_bounds, region_bounds = fit_region_winsor_bounds(df)
        y_target = apply_region_winsor(y_raw, df["region"], global_bounds, region_bounds)
        mask = pd.Series(True, index=df.index)
    elif variant == "floor500":
        y_target = y_raw.clip(lower=LOW_SALARY_FLOOR)
        mask = pd.Series(True, index=df.index)
    elif variant == "drop_lt500":
        mask = y_raw >= LOW_SALARY_FLOOR
        df = df.loc[mask].reset_index(drop=True)
        y_raw = y_raw.loc[mask].reset_index(drop=True)
        y_target = y_raw.copy()
        mask = pd.Series(True, index=df.index)
    else:
        raise ValueError(f"Unknown target variant: {variant}")

    y_log = safe_log_salary(y_target).reset_index(drop=True)
    y_raw_metric = pd.Series(y_target).astype(float).reset_index(drop=True)
    return df.reset_index(drop=True), y_log, y_raw_metric


for variant in ["raw", "winsorized", "floor500", "drop_lt500"]:
    dft, ylog, yraw_metric = get_training_target_variant(df_train, variant)
    print(f"{variant:12s} | rows={len(dft):4d} | log median={ylog.median():.3f} | raw metric median={yraw_metric.median():,.0f}")


raw          | rows=1758 | log median=10.642 | raw metric median=41,854
winsorized   | rows=1758 | log median=10.642 | raw metric median=41,854
floor500     | rows=1758 | log median=10.642 | raw metric median=41,854
drop_lt500   | rows=1701 | log median=10.680 | raw metric median=43,489


## 5. SVR-specific feature engineering class

This preprocessor is different from the previous broad feature engineering file.

For RBF SVR we test lower-dimensional feature profiles because RBF similarity is distance-based and can suffer from too many sparse binary indicators.

Important anti-leakage rule: the preprocessor is fitted only on the training data passed to it. Region target encoding, medians, rare categories, top multi-select items, and scaling are all train-fitted.


In [5]:
# ===============================================================
# 5. SVR-specific feature engineering
# ===============================================================
MULTI_SEP = ";"

ORDINAL_MAPS = {
    "age.group": {"18-24": 0, "25-34": 1, "35-44": 2, "45-54": 3, "55+": 4},
    "education": {
        "Primary/elementary school": 0,
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
        "Some college/university study without earning a degree": 2,
        "Associate degree (A.A., A.S., etc.)": 3,
        "Something else": 3,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 4,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    },
    "company.size": {
        "Just me - I am a freelancer, sole proprietor, etc.": 0,
        "2 to 9 employees": 1,
        "10 to 19 employees": 2,
        "20 to 99 employees": 3,
        "100 to 499 employees": 4,
        "500 to 999 employees": 5,
        "1,000 to 4,999 employees": 6,
        "5,000 to 9,999 employees": 7,
        "10,000 or more employees": 8,
    },
    "tech.purchase.influence": {
        "I have little or no influence": 0,
        "I have some influence": 1,
        "I have a great deal of influence": 2,
    },
    "ai.sentiment": {
        "Very unfavorable": 0,
        "Unfavorable": 1,
        "Indifferent": 2,
        "Unsure": 2,
        "Favorable": 3,
        "Very favorable": 4,
    },
    "ai.trust": {
        "Highly distrust": 0,
        "Somewhat distrust": 1,
        "Neither trust nor distrust": 2,
        "Somewhat trust": 3,
        "Highly trust": 4,
    },
    "ai.complex.rating": {
        "Very poor at handling complex tasks": 0,
        "Bad at handling complex tasks": 1,
        "Neither good or bad at handling complex tasks": 2,
        "Good, but not great at handling complex tasks": 3,
        "Very well at handling complex tasks": 4,
    },
    "daily.search.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
    "daily.answer.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
}

QUASI_ORDINAL_MAPS = {
    "is.dev.professional": {
        "I am a developer by profession": 1,
        "I am not primarily a developer, but I write code sometimes as part of my work/studies": 0,
    },
    "people.manager": {"People manager": 1, "Individual contributor": 0},
    "uses.ai": {"Yes": 2, "No, but I plan to soon": 1, "No, and I don't plan to": 0},
}

# build.vs.buy is intentionally one-hot encoded, not ordinal.
ONEHOT_BASE_COLS = [
    "region",
    "employment.type",
    "work.location",
    "dev.role",
    "industry",
    "cloud.hosting",
    "first.help.source",
    "ai.job.threat",
    "build.vs.buy",
]

MULTI_SELECT_COLS = [
    "prog.languages",
    "databases",
    "cloud.platforms",
    "web.frameworks",
    "other.tech",
    "dev.tools",
    "dev.environments",
    "personal.os",
    "work.os",
    "project.mgmt.tools",
    "comm.tools",
    "ai.search.tools",
    "ai.tools.used",
    "side.coding",
    "how.learned.coding",
]


def parse_multiselect_cell(value, sep: str = MULTI_SEP) -> List[str]:
    if pd.isna(value):
        return []
    value = normalize_text_value(value)
    if value == "":
        return []
    return [normalize_text_value(item) for item in str(value).split(sep) if normalize_text_value(item) != ""]


def sanitize_feature_name(name: str) -> str:
    name = normalize_text_value(name)
    name = str(name).lower()
    replacements = {
        " ": "_", "/": "_", ".": "_", "(": "", ")": "", ",": "",
        "'": "", "-": "_", "+": "plus", "#": "sharp", ":": "", "!": "",
        "&": "and", "[": "", "]": "",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r"[^a-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


@dataclass
class SVRRBFPreprocessor:
    # SVR-specific leakage-safe preprocessing.
    multi_top_n: int = 5
    include_region_target: bool = True
    include_region_dummies: bool = True
    include_experience_years: bool = False
    include_numeric_squares: bool = False
    missing_flag_threshold: float = 0.10
    rare_min_count_nominal: int = 20
    rare_min_count_region: int = 10
    target_encoding_smoothing: float = 20.0

    missing_indicator_cols_: List[str] = field(default_factory=list)
    numeric_medians_: Dict[str, float] = field(default_factory=dict)
    ordinal_fallbacks_: Dict[str, float] = field(default_factory=dict)
    rare_keep_values_: Dict[str, List[str]] = field(default_factory=dict)
    multi_top_items_: Dict[str, List[str]] = field(default_factory=dict)
    region_target_map_: Dict[str, float] = field(default_factory=dict)
    region_target_global_: float = 0.0
    feature_cols_: List[str] = field(default_factory=list)
    zero_variance_cols_: List[str] = field(default_factory=list)
    scaler_: Optional[StandardScaler] = None

    def _onehot_cols(self) -> List[str]:
        cols = ONEHOT_BASE_COLS.copy()
        if not self.include_region_dummies and "region" in cols:
            cols.remove("region")
        return cols

    def _numeric_cols(self) -> List[str]:
        cols = ["coding.years.professional", "job.satisfaction", "years.before.professional"]
        if self.include_experience_years:
            cols.insert(1, "experience.years")
        if self.include_numeric_squares:
            cols.extend([c + "_sq" for c in ["coding.years.professional"] + (["experience.years"] if self.include_experience_years else [])])
        return cols

    def fit(self, X_raw: pd.DataFrame, y_log: pd.Series) -> "SVRRBFPreprocessor":
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[c for c in [TARGET] if c in X.columns])

        onehot_cols = set(self._onehot_cols())
        missing_rates = X.isna().mean()
        self.missing_indicator_cols_ = [
            col for col, rate in missing_rates.items()
            if rate >= self.missing_flag_threshold and rate > 0 and col not in onehot_cols
        ]

        X_num = self._engineer_numeric(X)
        self.numeric_medians_ = {}
        for col in self._numeric_cols():
            if col in X_num.columns and X_num[col].notna().any():
                self.numeric_medians_[col] = float(X_num[col].median())
            else:
                self.numeric_medians_[col] = 0.0

        self.ordinal_fallbacks_ = {}
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                self.ordinal_fallbacks_[col] = float(mapped.median()) if mapped.notna().any() else 0.0

        self.rare_keep_values_ = {}
        for col in self._onehot_cols():
            if col not in X.columns:
                continue
            filled = X[col].fillna("__Missing__").astype(str)
            min_count = self.rare_min_count_region if col == "region" else self.rare_min_count_nominal
            counts = filled.value_counts(dropna=False)
            keep = counts[counts >= min_count].index.tolist()
            if not keep and len(counts) > 0:
                keep = [counts.index[0]]
            self.rare_keep_values_[col] = [str(v) for v in keep]

        self.multi_top_items_ = {}
        for col in MULTI_SELECT_COLS:
            if col not in X.columns:
                continue
            counter = Counter()
            for value in X[col]:
                counter.update(parse_multiselect_cell(value))
            if self.multi_top_n > 0:
                self.multi_top_items_[col] = [item for item, _ in counter.most_common(self.multi_top_n)]
            else:
                self.multi_top_items_[col] = []

        if self.include_region_target and "region" in X.columns:
            y = pd.Series(y_log).astype(float).reset_index(drop=True)
            regions = X["region"].fillna("__Missing__").astype(str).reset_index(drop=True)
            self.region_target_global_ = float(y.mean())
            enc_df = pd.DataFrame({"region": regions, "y": y})
            stats = enc_df.groupby("region")["y"].agg(["mean", "count"])
            k = float(self.target_encoding_smoothing)
            stats["smoothed"] = (stats["mean"] * stats["count"] + self.region_target_global_ * k) / (stats["count"] + k)
            self.region_target_map_ = stats["smoothed"].to_dict()
        else:
            self.region_target_global_ = 0.0
            self.region_target_map_ = {}

        X_unscaled = self._build_unscaled(X)
        self.zero_variance_cols_ = X_unscaled.columns[X_unscaled.nunique(dropna=False) <= 1].tolist()
        X_unscaled = X_unscaled.drop(columns=self.zero_variance_cols_, errors="ignore")
        self.feature_cols_ = X_unscaled.columns.tolist()
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X_unscaled)
        return self

    def transform(self, X_raw: pd.DataFrame, scaled: bool = True) -> pd.DataFrame:
        if not self.feature_cols_ or self.scaler_ is None:
            raise RuntimeError("Preprocessor must be fitted before transform.")
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[c for c in [TARGET] if c in X.columns])
        X_unscaled = self._build_unscaled(X)
        X_unscaled = X_unscaled.drop(columns=self.zero_variance_cols_, errors="ignore")
        X_unscaled = self._align_to_features(X_unscaled, self.feature_cols_)
        if not scaled:
            return X_unscaled
        X_scaled = pd.DataFrame(self.scaler_.transform(X_unscaled), columns=self.feature_cols_, index=X_unscaled.index)
        return X_scaled

    def fit_transform(self, X_raw: pd.DataFrame, y_log: pd.Series, scaled: bool = True) -> pd.DataFrame:
        self.fit(X_raw, y_log)
        return self.transform(X_raw, scaled=scaled)

    def _build_unscaled(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        pieces = []

        # Missing indicators.
        miss = pd.DataFrame(index=X.index)
        for col in self.missing_indicator_cols_:
            if col in X.columns:
                miss[f"{col}_missing"] = X[col].isna().astype(float)
        if "company.size" in X.columns:
            miss["company.size_unknown"] = (X["company.size"] == "I don't know").astype(float)
        if not miss.empty:
            pieces.append(miss)

        # Numeric features.
        X_num = self._engineer_numeric(X)
        for col, median in self.numeric_medians_.items():
            if col not in X_num.columns:
                X_num[col] = median
            else:
                X_num[col] = X_num[col].fillna(median)
        numeric_cols = self._numeric_cols()
        for col in numeric_cols:
            if col not in X_num.columns:
                X_num[col] = self.numeric_medians_.get(col, 0.0)
        pieces.append(X_num[numeric_cols].astype(float))

        # Ordinal features.
        ord_df = pd.DataFrame(index=X.index)
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                ord_df[col] = mapped.fillna(self.ordinal_fallbacks_.get(col, 0.0)).astype(float)
        if not ord_df.empty:
            pieces.append(ord_df)

        # Region target encoding.
        if self.include_region_target and "region" in X.columns:
            reg = X["region"].fillna("__Missing__").astype(str)
            enc = reg.map(self.region_target_map_).fillna(self.region_target_global_).astype(float)
            pieces.append(pd.DataFrame({"region_target_enc": enc}, index=X.index))

        # One-hot features.
        ohe_source = pd.DataFrame(index=X.index)
        for col in self._onehot_cols():
            if col not in X.columns:
                continue
            filled = X[col].fillna("__Missing__").astype(str)
            keep = set(self.rare_keep_values_.get(col, []))
            ohe_source[col] = filled.where(filled.isin(keep), "__Rare__")
        if not ohe_source.empty:
            dummies = pd.get_dummies(ohe_source, columns=ohe_source.columns.tolist(), prefix_sep="__", drop_first=True, dtype=float)
            pieces.append(dummies)

        # Multi-select: counts for all, plus top-N indicators if configured.
        multi = pd.DataFrame(index=X.index)
        for col in MULTI_SELECT_COLS:
            if col not in X.columns:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            prefix = col.replace(".", "_")
            multi[f"{prefix}_count"] = [len(items) for items in parsed]
            for item in self.multi_top_items_.get(col, []):
                multi[f"{prefix}__{sanitize_feature_name(item)}"] = [int(item in items) for items in parsed]
        if not multi.empty:
            pieces.append(multi.astype(float))

        X_out = pd.concat(pieces, axis=1) if pieces else pd.DataFrame(index=X.index)
        X_out = X_out.loc[:, ~X_out.columns.duplicated()].fillna(0).astype(float)
        return X_out

    def _engineer_numeric(self, X: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=X.index)
        out["coding.years.professional"] = pd.to_numeric(X.get("coding.years.professional", pd.Series(np.nan, index=X.index)), errors="coerce")
        out["experience.years"] = pd.to_numeric(X.get("experience.years", pd.Series(np.nan, index=X.index)), errors="coerce")
        out["job.satisfaction"] = pd.to_numeric(X.get("job.satisfaction", pd.Series(np.nan, index=X.index)), errors="coerce")
        total = pd.to_numeric(X.get("coding.years.total", pd.Series(np.nan, index=X.index)), errors="coerce")
        prof = out["coding.years.professional"]
        out["years.before.professional"] = (total - prof).clip(lower=0)
        out["coding.years.professional_sq"] = out["coding.years.professional"] ** 2
        out["experience.years_sq"] = out["experience.years"] ** 2
        return out

    @staticmethod
    def _align_to_features(X: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
        X = X.copy()
        for col in feature_cols:
            if col not in X.columns:
                X[col] = 0.0
        extra = [c for c in X.columns if c not in feature_cols]
        if extra:
            X = X.drop(columns=extra)
        return X[feature_cols].fillna(0).astype(float)

print("SVR-specific preprocessor ready.")


SVR-specific preprocessor ready.


## 6. Define SVR feature profiles and parameter grids

We deliberately test feature profiles that are more suitable for distance/kernel models.

The most important profiles are:

- `counts_te`: multi-select counts + region target encoding,
- `top5_te`: top 5 binary indicators per multi-select column + region target encoding,
- `top10_te`: top 10 binary indicators per multi-select column + region target encoding,
- `top10_te_no_region_dummies`: target encoding but no region one-hot dummies, reducing sparse dimensions.


In [6]:
# ===============================================================
# 6. Feature profiles and RBF SVR parameter grid
# ===============================================================
feature_profiles = [
    {
        "name": "counts_te",
        "multi_top_n": 0,
        "include_region_target": True,
        "include_region_dummies": True,
        "include_experience_years": False,
        "include_numeric_squares": False,
    },
    {
        "name": "top5_te",
        "multi_top_n": 5,
        "include_region_target": True,
        "include_region_dummies": True,
        "include_experience_years": False,
        "include_numeric_squares": False,
    },
    {
        "name": "top10_te",
        "multi_top_n": 10,
        "include_region_target": True,
        "include_region_dummies": True,
        "include_experience_years": False,
        "include_numeric_squares": False,
    },
    {
        "name": "top10_te_no_region_dummies",
        "multi_top_n": 10,
        "include_region_target": True,
        "include_region_dummies": False,
        "include_experience_years": False,
        "include_numeric_squares": False,
    },
]

if not FAST_MODE:
    feature_profiles.extend([
        {
            "name": "top5_no_te",
            "multi_top_n": 5,
            "include_region_target": False,
            "include_region_dummies": True,
            "include_experience_years": False,
            "include_numeric_squares": False,
        },
        {
            "name": "top5_te_with_experience",
            "multi_top_n": 5,
            "include_region_target": True,
            "include_region_dummies": True,
            "include_experience_years": True,
            "include_numeric_squares": False,
        },
    ])

# Focused RBF grid around the previously good area.
if FAST_MODE:
    svr_param_grid = list(ParameterGrid({
        "C": [0.5, 1.0, 2.0],
        "gamma": ["scale", 0.0003],
        "epsilon": [0.05, 0.1],
    }))
    target_variants = ["raw", "winsorized", "floor500", "drop_lt500"]
else:
    svr_param_grid = list(ParameterGrid({
        "C": [0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0],
        "gamma": ["scale", 0.0001, 0.0002, 0.0003, 0.0005, 0.001],
        "epsilon": [0.03, 0.05, 0.1, 0.15, 0.2],
    }))
    target_variants = ["raw", "winsorized", "floor500", "drop_lt500"]

print("Number of feature profiles:", len(feature_profiles))
print("Number of SVR parameter combinations:", len(svr_param_grid))
print("Number of target variants:", len(target_variants))
print("Approx validation fits:", len(feature_profiles) * len(svr_param_grid) * len(target_variants))


Number of feature profiles: 4
Number of SVR parameter combinations: 12
Number of target variants: 4
Approx validation fits: 192


## 7. Evaluation helpers

We evaluate using **raw USD RMSE** because this is the competition metric.

Calibration options:

- `none`: simply `exp(pred_log)`,
- `train_smearing`: Duan-style correction from training residuals,
- conservative fixed multipliers such as `fixed_1.03`,
- `validation_multiplier`: best multiplier on validation; useful diagnostically, but potentially overfit.


In [7]:
# ===============================================================
# 7. Evaluation helpers
# ===============================================================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_exp_to_usd(pred_log: np.ndarray, factor: float = 1.0) -> np.ndarray:
    pred = np.exp(np.asarray(pred_log, dtype=float)) * float(factor)
    return np.clip(pred, PREDICTION_MIN_USD, PREDICTION_MAX_USD)


def compute_validation_multiplier(y_val_raw: pd.Series, pred_val_log: np.ndarray, grid=None) -> Tuple[float, float]:
    if grid is None:
        grid = np.linspace(0.85, 1.25, 81)
    scores = [rmse(y_val_raw, safe_exp_to_usd(pred_val_log, factor=f)) for f in grid]
    best_idx = int(np.argmin(scores))
    return float(grid[best_idx]), float(scores[best_idx])


def evaluate_corrections(y_train_log, pred_train_log, y_val_raw, pred_val_log) -> List[Dict[str, Any]]:
    residuals = np.asarray(y_train_log) - np.asarray(pred_train_log)
    smearing = float(np.mean(np.exp(residuals)))
    corrections = []
    # Safer corrections.
    for name, factor in [
        ("none", 1.0),
        ("train_smearing", smearing),
        ("fixed_1.03", 1.03),
        ("fixed_1.05", 1.05),
        ("fixed_1.08", 1.08),
        ("fixed_1.10", 1.10),
    ]:
        corrections.append({
            "correction": name,
            "factor": float(factor),
            "val_rmse": rmse(y_val_raw, safe_exp_to_usd(pred_val_log, factor=factor)),
            "leaderboard_safe": name != "validation_multiplier",
        })
    val_factor, val_score = compute_validation_multiplier(y_val_raw, pred_val_log)
    corrections.append({
        "correction": "validation_multiplier",
        "factor": val_factor,
        "val_rmse": val_score,
        "leaderboard_safe": False,
    })
    return corrections


def make_model(params: Dict[str, Any]) -> SVR:
    return SVR(kernel="rbf", cache_size=1000, **params)

print("Evaluation helpers ready.")


Evaluation helpers ready.


## 8. Stage 1 — Validation search for RBF SVR only

This is a focused validation search. For each candidate, we:

1. create the selected training target variant,
2. fit SVR-specific preprocessing on that training subset only,
3. train RBF SVR,
4. predict validation,
5. evaluate all calibration options.

Partial results are saved after each candidate.


In [8]:
# ===============================================================
# 8. Stage 1 validation search
# ===============================================================
def fit_predict_candidate_validation(profile: Dict[str, Any], target_variant: str, svr_params: Dict[str, Any]) -> Tuple[Dict[str, Any], Any]:
    # Prepare target variant and possibly reduced training rows.
    dft, y_train_log, y_train_metric_raw = get_training_target_variant(df_train, target_variant)
    X_train_raw = dft.drop(columns=[TARGET], errors="ignore")
    X_val_raw = df_val.drop(columns=[TARGET], errors="ignore")

    # Build preprocessor from profile.
    preproc_kwargs = {k: v for k, v in profile.items() if k != "name"}
    preprocessor = SVRRBFPreprocessor(**preproc_kwargs)
    X_train = preprocessor.fit_transform(X_train_raw, y_train_log, scaled=True)
    X_val = preprocessor.transform(X_val_raw, scaled=True)

    model = make_model(svr_params)
    start = time.time()
    model.fit(X_train, y_train_log)
    fit_seconds = time.time() - start

    pred_train_log = np.asarray(model.predict(X_train), dtype=float)
    pred_val_log = np.asarray(model.predict(X_val), dtype=float)

    meta = {
        "feature_profile": profile["name"],
        "target_variant": target_variant,
        "svr_params": json.dumps(svr_params),
        "C": svr_params["C"],
        "gamma": str(svr_params["gamma"]),
        "epsilon": svr_params["epsilon"],
        "n_features": X_train.shape[1],
        "n_train_rows": X_train.shape[0],
        "fit_seconds": fit_seconds,
    }
    fitted_objects = {
        "preprocessor": preprocessor,
        "model": model,
        "X_train": X_train,
        "X_val": X_val,
        "y_train_log": y_train_log,
        "pred_train_log": pred_train_log,
        "pred_val_log": pred_val_log,
    }
    return meta, fitted_objects


validation_records = []
total = len(feature_profiles) * len(target_variants) * len(svr_param_grid)
counter = 0
start_all = time.time()

for profile in feature_profiles:
    for target_variant in target_variants:
        for params in svr_param_grid:
            counter += 1
            print(f"[{counter:03d}/{total:03d}] profile={profile['name']} | target={target_variant} | params={params}")
            try:
                meta, objs = fit_predict_candidate_validation(profile, target_variant, params)
                corr_rows = evaluate_corrections(
                    objs["y_train_log"], objs["pred_train_log"], y_val_raw=df_val[TARGET].astype(float), pred_val_log=objs["pred_val_log"]
                )
                for corr in corr_rows:
                    row = {**meta, **corr}
                    validation_records.append(row)
                best_corr = min(corr_rows, key=lambda d: d["val_rmse"])
                best_safe = min([r for r in corr_rows if r["leaderboard_safe"]], key=lambda d: d["val_rmse"])
                print(f"    best={best_corr['correction']} RMSE={best_corr['val_rmse']:,.0f} | best_safe={best_safe['correction']} RMSE={best_safe['val_rmse']:,.0f}")
            except Exception as exc:
                print("    FAILED:", exc)
                validation_records.append({
                    "feature_profile": profile["name"],
                    "target_variant": target_variant,
                    "svr_params": json.dumps(params),
                    "C": params.get("C"),
                    "gamma": str(params.get("gamma")),
                    "epsilon": params.get("epsilon"),
                    "correction": "failed",
                    "factor": np.nan,
                    "val_rmse": np.nan,
                    "leaderboard_safe": False,
                    "error": str(exc),
                })
            pd.DataFrame(validation_records).to_csv(OUTPUT_DIR / "svr_rbf_validation_search_partial.csv", index=False)

validation_results = pd.DataFrame(validation_records).dropna(subset=["val_rmse"]).sort_values("val_rmse")
validation_results.to_csv(OUTPUT_DIR / "svr_rbf_validation_search.csv", index=False)

print("\nSearch finished in minutes:", (time.time() - start_all) / 60)
print("Best validation rows:")
display(validation_results.head(30))


[001/192] profile=counts_te | target=raw | params={'C': 0.5, 'epsilon': 0.05, 'gamma': 'scale'}
    best=validation_multiplier RMSE=38,553 | best_safe=fixed_1.08 RMSE=38,553
[002/192] profile=counts_te | target=raw | params={'C': 0.5, 'epsilon': 0.05, 'gamma': 0.0003}
    best=validation_multiplier RMSE=39,470 | best_safe=fixed_1.10 RMSE=39,488
[003/192] profile=counts_te | target=raw | params={'C': 0.5, 'epsilon': 0.1, 'gamma': 'scale'}
    best=validation_multiplier RMSE=38,556 | best_safe=fixed_1.08 RMSE=38,557
[004/192] profile=counts_te | target=raw | params={'C': 0.5, 'epsilon': 0.1, 'gamma': 0.0003}
    best=validation_multiplier RMSE=39,467 | best_safe=train_smearing RMSE=39,488
[005/192] profile=counts_te | target=raw | params={'C': 1.0, 'epsilon': 0.05, 'gamma': 'scale'}
    best=validation_multiplier RMSE=39,132 | best_safe=fixed_1.05 RMSE=39,133
[006/192] profile=counts_te | target=raw | params={'C': 1.0, 'epsilon': 0.05, 'gamma': 0.0003}
    best=validation_multiplier RMSE

,feature_profile,target_variant,svr_params,C,gamma,epsilon,n_features,n_train_rows,fit_seconds,correction,factor,val_rmse,leaderboard_safe
440,top5_te,winsorized,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.733341,validation_multiplier,1.085000,36970.354096,False
438,top5_te,winsorized,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.733341,fixed_1.08,1.080000,36971.084166,True
356,top5_te,raw,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.606186,validation_multiplier,1.085000,36971.828119,False
354,top5_te,raw,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.606186,fixed_1.08,1.080000,36972.462182,True
524,top5_te,floor500,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,1.085236,validation_multiplier,1.085000,36973.314339,False
522,top5_te,floor500,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,1.085236,fixed_1.08,1.080000,36973.932582,True
439,top5_te,winsorized,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.733341,fixed_1.10,1.100000,36977.651457,True
355,top5_te,raw,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,0.606186,fixed_1.10,1.100000,36979.418027,True
523,top5_te,floor500,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.10,184,1758,1.085236,fixed_1.10,1.100000,36980.951813,True
426,top5_te,winsorized,"{""C"": 0.5, ""epsilon"": 0.05, ""gamma"": ""scale""}",0.5,scale,0.05,184,1758,0.769632,validation_multiplier,1.070000,36994.842923,False


## 9. Stage 2 — Fold-safe CV for top candidates

The previous broad notebook used precomputed matrices, which means CV was only approximate. Here, for top RBF candidates, we run a **fold-safe CV**:

- preprocessing is fitted separately inside each fold,
- region target encoding is fitted only on the fold training part,
- scaling is fitted only on the fold training part.

This is slower, so we run it only on the top validation candidates.


In [9]:
# ===============================================================
# 9. Fold-safe CV for top candidates
# ===============================================================
def get_profile_by_name(name: str) -> Dict[str, Any]:
    for p in feature_profiles:
        if p["name"] == name:
            return p
    raise KeyError(name)


def parse_svr_params(params_json: str) -> Dict[str, Any]:
    if isinstance(params_json, dict):
        return params_json
    return json.loads(params_json)


def fold_safe_cv_for_candidate(row: pd.Series) -> Dict[str, Any]:
    profile = get_profile_by_name(row["feature_profile"])
    target_variant = row["target_variant"]
    svr_params = parse_svr_params(row["svr_params"])
    correction = row["correction"]
    fixed_factor = float(row["factor"])

    # Create target variant first, then CV inside its training rows.
    dft, y_log_all, y_raw_metric_all = get_training_target_variant(df_train, target_variant)
    regions = dft["region"].fillna("__Missing__").astype(str).reset_index(drop=True)
    strata = make_region_strata(regions)
    counts = strata.value_counts()
    if len(counts) >= 2 and counts.min() >= N_SPLITS_CV:
        splitter = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
        split_iter = splitter.split(dft, strata)
    else:
        splitter = KFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
        split_iter = splitter.split(dft)

    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(split_iter, start=1):
        fold_train = dft.iloc[tr_idx].reset_index(drop=True)
        fold_val = dft.iloc[va_idx].reset_index(drop=True)
        y_fold_train_log = y_log_all.iloc[tr_idx].reset_index(drop=True)
        y_fold_val_raw = fold_val[TARGET].astype(float).reset_index(drop=True)

        preproc_kwargs = {k: v for k, v in profile.items() if k != "name"}
        preprocessor = SVRRBFPreprocessor(**preproc_kwargs)
        X_fold_train = preprocessor.fit_transform(fold_train.drop(columns=[TARGET], errors="ignore"), y_fold_train_log, scaled=True)
        X_fold_val = preprocessor.transform(fold_val.drop(columns=[TARGET], errors="ignore"), scaled=True)

        model = make_model(svr_params)
        model.fit(X_fold_train, y_fold_train_log)
        pred_train_log = np.asarray(model.predict(X_fold_train), dtype=float)
        pred_val_log = np.asarray(model.predict(X_fold_val), dtype=float)

        if correction == "none":
            factor = 1.0
        elif correction == "train_smearing":
            factor = float(np.mean(np.exp(np.asarray(y_fold_train_log) - pred_train_log)))
        elif correction.startswith("fixed_"):
            factor = fixed_factor
        elif correction == "validation_multiplier":
            # Not a fold-safe correction; optimize within each fold only as an optimistic diagnostic.
            factor, _ = compute_validation_multiplier(y_fold_val_raw, pred_val_log)
        else:
            factor = fixed_factor

        score = rmse(y_fold_val_raw, safe_exp_to_usd(pred_val_log, factor=factor))
        fold_scores.append(score)

    return {
        "cv_rmse_mean": float(np.mean(fold_scores)),
        "cv_rmse_std": float(np.std(fold_scores, ddof=1)),
        "cv_fold_scores": json.dumps([float(s) for s in fold_scores]),
    }

# To avoid repeated duplicates, take best row per base config/correction.
if 'validation_results' not in globals():
    validation_results = pd.read_csv(OUTPUT_DIR / "svr_rbf_validation_search.csv")

# Prioritize leaderboard-safe rows and also keep top absolute rows.
top_abs = validation_results.sort_values("val_rmse").head(N_TOP_FOR_CV)
top_safe = validation_results[validation_results["leaderboard_safe"]].sort_values("val_rmse").head(N_TOP_FOR_CV)
top_for_cv = pd.concat([top_abs, top_safe], ignore_index=True).drop_duplicates(
    subset=["feature_profile", "target_variant", "svr_params", "correction", "factor"]
).head(N_TOP_FOR_CV * 2)

cv_records = []
for i, (_, row) in enumerate(top_for_cv.iterrows(), start=1):
    print(f"[{i}/{len(top_for_cv)}] CV: profile={row['feature_profile']} target={row['target_variant']} correction={row['correction']} val_rmse={row['val_rmse']:,.0f}")
    try:
        cv_info = fold_safe_cv_for_candidate(row)
        rec = row.to_dict()
        rec.update(cv_info)
        cv_records.append(rec)
        print(f"    CV RMSE={cv_info['cv_rmse_mean']:,.0f} ± {cv_info['cv_rmse_std']:,.0f}")
    except Exception as exc:
        rec = row.to_dict()
        rec.update({"cv_rmse_mean": np.nan, "cv_rmse_std": np.nan, "cv_error": str(exc)})
        cv_records.append(rec)
        print("    FAILED:", exc)
    pd.DataFrame(cv_records).to_csv(OUTPUT_DIR / "svr_rbf_top_candidates_cv_partial.csv", index=False)

cv_results = pd.DataFrame(cv_records)
# A pragmatic robustness score. Validation is still primary; CV is a stability check.
cv_results["robust_score"] = cv_results["val_rmse"] + 0.10 * cv_results["cv_rmse_std"].fillna(0)
cv_results = cv_results.sort_values(["leaderboard_safe", "robust_score"], ascending=[False, True])
cv_results.to_csv(OUTPUT_DIR / "svr_rbf_top_candidates_cv.csv", index=False)

display(cv_results[[
    "feature_profile", "target_variant", "C", "gamma", "epsilon", "correction", "factor",
    "val_rmse", "cv_rmse_mean", "cv_rmse_std", "robust_score", "n_features", "n_train_rows", "leaderboard_safe"
]].head(30))


[1/14] CV: profile=top5_te target=winsorized correction=validation_multiplier val_rmse=36,970
    CV RMSE=96,545 ± 88,711
[2/14] CV: profile=top5_te target=winsorized correction=fixed_1.08 val_rmse=36,971
    CV RMSE=96,743 ± 88,744
[3/14] CV: profile=top5_te target=raw correction=validation_multiplier val_rmse=36,972
    CV RMSE=96,545 ± 88,711
[4/14] CV: profile=top5_te target=raw correction=fixed_1.08 val_rmse=36,972
    CV RMSE=96,743 ± 88,744
[5/14] CV: profile=top5_te target=floor500 correction=validation_multiplier val_rmse=36,973
    CV RMSE=96,549 ± 88,710
[6/14] CV: profile=top5_te target=floor500 correction=fixed_1.08 val_rmse=36,974
    CV RMSE=96,747 ± 88,743
[7/14] CV: profile=top5_te target=winsorized correction=fixed_1.10 val_rmse=36,978
    CV RMSE=96,690 ± 88,752
[8/14] CV: profile=top5_te target=raw correction=fixed_1.10 val_rmse=36,979
    CV RMSE=96,691 ± 88,752
[9/14] CV: profile=top5_te target=floor500 correction=fixed_1.10 val_rmse=36,981
    CV RMSE=96,694 ± 88

,feature_profile,target_variant,C,gamma,epsilon,correction,factor,val_rmse,cv_rmse_mean,cv_rmse_std,robust_score,n_features,n_train_rows,leaderboard_safe
1,top5_te,winsorized,0.5,scale,0.10,fixed_1.08,1.080,36971.084166,96743.201968,88744.208184,45845.504985,184,1758,True
3,top5_te,raw,0.5,scale,0.10,fixed_1.08,1.080,36972.462182,96743.446170,88743.886077,45846.850790,184,1758,True
5,top5_te,floor500,0.5,scale,0.10,fixed_1.08,1.080,36973.932582,96746.818683,88742.783544,45848.210936,184,1758,True
6,top5_te,winsorized,0.5,scale,0.10,fixed_1.10,1.100,36977.651457,96690.450135,88752.058786,45852.857336,184,1758,True
7,top5_te,raw,0.5,scale,0.10,fixed_1.10,1.100,36979.418027,96690.778824,88751.691809,45854.587208,184,1758,True
8,top5_te,floor500,0.5,scale,0.10,fixed_1.10,1.100,36980.951813,96694.265598,88750.539876,45856.005801,184,1758,True
10,top5_te,winsorized,0.5,scale,0.05,fixed_1.08,1.080,36997.324763,96661.671517,88739.703116,45871.295074,184,1758,True
11,top5_te,raw,0.5,scale,0.05,fixed_1.08,1.080,36998.655372,96661.472296,88739.647149,45872.620087,184,1758,True
12,top5_te,floor500,0.5,scale,0.05,fixed_1.08,1.080,36999.973541,96664.479962,88738.609720,45873.834513,184,1758,True
13,top5_te,winsorized,0.5,scale,0.10,fixed_1.05,1.050,37008.653912,96846.694976,88721.500258,45880.803938,184,1758,True


## 10. Internal-test diagnostic for selected candidates

Because the previous notebook already revealed a large validation/internal-test gap, this section evaluates a few selected candidates on the internal test **for diagnosis**.

Do not keep tuning indefinitely on internal test. Use this to understand whether a candidate is more stable than the previous submission.


In [10]:
# ===============================================================
# 10. Internal-test diagnostic for selected candidates
# ===============================================================
def fit_candidate_on_internal_train_and_predict(row: pd.Series, X_new_raw: pd.DataFrame):
    profile = get_profile_by_name(row["feature_profile"])
    target_variant = row["target_variant"]
    svr_params = parse_svr_params(row["svr_params"])

    dft, y_train_log, _ = get_training_target_variant(df_train, target_variant)
    preproc_kwargs = {k: v for k, v in profile.items() if k != "name"}
    preprocessor = SVRRBFPreprocessor(**preproc_kwargs)
    X_train = preprocessor.fit_transform(dft.drop(columns=[TARGET], errors="ignore"), y_train_log, scaled=True)
    X_new = preprocessor.transform(X_new_raw, scaled=True)

    model = make_model(svr_params)
    model.fit(X_train, y_train_log)
    pred_train_log = np.asarray(model.predict(X_train), dtype=float)
    pred_new_log = np.asarray(model.predict(X_new), dtype=float)

    correction = row["correction"]
    if correction == "none":
        factor = 1.0
    elif correction == "train_smearing":
        factor = float(np.mean(np.exp(np.asarray(y_train_log) - pred_train_log)))
    elif correction.startswith("fixed_"):
        factor = float(row["factor"])
    elif correction == "validation_multiplier":
        factor = float(row["factor"])
    else:
        factor = float(row["factor"])

    return pred_new_log, factor, model, preprocessor

if 'cv_results' not in globals():
    cv_results = pd.read_csv(OUTPUT_DIR / "svr_rbf_top_candidates_cv.csv")

# Evaluate top safe candidates and the top absolute validation candidate.
diagnostic_candidates = pd.concat([
    validation_results.sort_values("val_rmse").head(3),
    cv_results[cv_results["leaderboard_safe"]].sort_values("val_rmse").head(5),
], ignore_index=True).drop_duplicates(subset=["feature_profile", "target_variant", "svr_params", "correction", "factor"])

internal_records = []
for i, (_, row) in enumerate(diagnostic_candidates.iterrows(), start=1):
    print(f"[{i}/{len(diagnostic_candidates)}] internal diagnostic: {row['feature_profile']} | {row['target_variant']} | {row['correction']} | val={row['val_rmse']:,.0f}")
    pred_test_log, factor_used, _, _ = fit_candidate_on_internal_train_and_predict(
        row, df_internal_test.drop(columns=[TARGET], errors="ignore")
    )
    pred_test_usd = safe_exp_to_usd(pred_test_log, factor=factor_used)
    test_rmse = rmse(df_internal_test[TARGET].astype(float), pred_test_usd)
    test_mae = mean_absolute_error(df_internal_test[TARGET].astype(float), pred_test_usd)
    rec = row.to_dict()
    rec.update({
        "internal_test_rmse": test_rmse,
        "internal_test_mae": float(test_mae),
        "factor_used_internal": factor_used,
    })
    internal_records.append(rec)

internal_diag = pd.DataFrame(internal_records).sort_values("internal_test_rmse")
internal_diag.to_csv(OUTPUT_DIR / "svr_rbf_internal_test_diagnostic.csv", index=False)

display(internal_diag[[
    "feature_profile", "target_variant", "C", "gamma", "epsilon", "correction", "factor",
    "val_rmse", "internal_test_rmse", "internal_test_mae", "n_features", "leaderboard_safe"
]])


[1/7] internal diagnostic: top5_te | winsorized | validation_multiplier | val=36,970
[2/7] internal diagnostic: top5_te | winsorized | fixed_1.08 | val=36,971
[3/7] internal diagnostic: top5_te | raw | validation_multiplier | val=36,972
[4/7] internal diagnostic: top5_te | raw | fixed_1.08 | val=36,972
[5/7] internal diagnostic: top5_te | floor500 | fixed_1.08 | val=36,974
[6/7] internal diagnostic: top5_te | winsorized | fixed_1.10 | val=36,978
[7/7] internal diagnostic: top5_te | raw | fixed_1.10 | val=36,979


,feature_profile,target_variant,C,gamma,epsilon,correction,factor,val_rmse,internal_test_rmse,internal_test_mae,n_features,leaderboard_safe
3,top5_te,raw,0.5,scale,0.1,fixed_1.08,1.080,36972.462182,52761.399018,24561.342077,184,True
1,top5_te,winsorized,0.5,scale,0.1,fixed_1.08,1.080,36971.084166,52761.494182,24561.440183,184,True
2,top5_te,raw,0.5,scale,0.1,validation_multiplier,1.085,36971.828119,52765.033770,24595.405379,184,False
0,top5_te,winsorized,0.5,scale,0.1,validation_multiplier,1.085,36970.354096,52765.103457,24595.591563,184,False
4,top5_te,floor500,0.5,scale,0.1,fixed_1.08,1.080,36973.932582,52765.846558,24559.588798,184,True
5,top5_te,winsorized,0.5,scale,0.1,fixed_1.10,1.100,36977.651457,52782.166821,24709.460540,184,True
6,top5_te,raw,0.5,scale,0.1,fixed_1.10,1.100,36979.418027,52782.174977,24709.114659,184,True


## 11. Error analysis for the best diagnostic candidate

This section shows where the selected candidate fails: very low salaries, ordinary salaries, or very high salaries.


In [11]:
# ===============================================================
# 11. Error analysis for the best diagnostic candidate
# ===============================================================
if len(internal_diag) == 0:
    raise RuntimeError("No internal diagnostic results available.")

best_diag_row = internal_diag.iloc[0]
pred_test_log, factor_used, _, _ = fit_candidate_on_internal_train_and_predict(
    best_diag_row, df_internal_test.drop(columns=[TARGET], errors="ignore")
)
pred_test_usd = safe_exp_to_usd(pred_test_log, factor=factor_used)

y_test_raw = df_internal_test[TARGET].astype(float).reset_index(drop=True)
err_df = pd.DataFrame({
    "actual_salary": y_test_raw,
    "predicted_salary": pred_test_usd,
    "error": pred_test_usd - y_test_raw,
    "absolute_error": np.abs(pred_test_usd - y_test_raw),
    "region": df_internal_test["region"].reset_index(drop=True),
})
err_df["salary_bucket"] = pd.cut(
    err_df["actual_salary"],
    bins=[0, 500, 10_000, 30_000, 60_000, 100_000, 200_000, np.inf],
    labels=["<500", "500-10k", "10k-30k", "30k-60k", "60k-100k", "100k-200k", "200k+"],
    include_lowest=True,
)

bucket_summary = err_df.groupby("salary_bucket", observed=False).agg(
    n=("actual_salary", "size"),
    mean_actual=("actual_salary", "mean"),
    mean_pred=("predicted_salary", "mean"),
    rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
    mae=("absolute_error", "mean"),
).reset_index()

region_summary = err_df.groupby("region").agg(
    n=("actual_salary", "size"),
    mean_actual=("actual_salary", "mean"),
    mean_pred=("predicted_salary", "mean"),
    rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
    mae=("absolute_error", "mean"),
).sort_values("n", ascending=False).reset_index()

print("Best diagnostic candidate:")
display(pd.DataFrame([best_diag_row]))
print("Internal-test RMSE:", rmse(y_test_raw, pred_test_usd))
print("Internal-test MAE:", mean_absolute_error(y_test_raw, pred_test_usd))

display(bucket_summary)
display(region_summary.head(20))

err_df.to_csv(OUTPUT_DIR / "best_diagnostic_internal_test_predictions.csv", index=False)
bucket_summary.to_csv(OUTPUT_DIR / "best_diagnostic_error_by_salary_bucket.csv", index=False)
region_summary.to_csv(OUTPUT_DIR / "best_diagnostic_error_by_region.csv", index=False)


Best diagnostic candidate:


,feature_profile,target_variant,svr_params,C,gamma,epsilon,n_features,n_train_rows,fit_seconds,correction,factor,val_rmse,leaderboard_safe,cv_rmse_mean,cv_rmse_std,cv_fold_scores,robust_score,internal_test_rmse,internal_test_mae,factor_used_internal
3,top5_te,raw,"{""C"": 0.5, ""epsilon"": 0.1, ""gamma"": ""scale""}",0.5,scale,0.1,184,1758,0.606186,fixed_1.08,1.08,36972.462182,True,96743.44617,88743.886077,"[254064.3080883638, 44599.59295063492, 47969.5...",45846.85079,52761.399018,24561.342077,1.08


Internal-test RMSE: 52761.399018461016
Internal-test MAE: 24561.34207660085


,salary_bucket,n,mean_actual,mean_pred,rmse,mae
0,<500,12,249.250000,30133.101917,30805.038720,29883.851917
1,500-10k,58,2718.844828,38526.083065,41069.305166,35807.238237
2,10k-30k,73,21180.684932,29891.188324,14551.410624,11185.520013
3,30k-60k,137,45077.379562,48170.109381,18955.155352,13859.156533
4,60k-100k,74,74173.378378,57154.257580,27486.274306,23792.433479
5,100k-200k,21,132373.000000,69906.445003,71652.170758,66337.561053
6,200k+,2,557293.000000,79681.171694,596599.422781,477611.828306


,region,n,mean_actual,mean_pred,rmse,mae
0,R17,166,32378.680723,35735.675075,31254.239802,23058.446670
1,R11,65,59985.738462,59484.913207,36150.299544,26260.961011
2,R13,32,43421.812500,50089.377017,28425.499689,19329.330970
3,R18,18,55454.166667,50247.418667,39394.937918,27356.846958
4,R03,17,98686.470588,50317.772465,203735.708201,65071.953623
5,R05,12,54507.166667,55652.357304,20740.832591,16859.247675
6,R06,11,43516.545455,46394.905395,28253.068960,25609.377406
7,R16,9,51722.666667,43253.073591,29696.553296,24032.003767
8,R09,9,52779.333333,53195.297178,13537.707675,11420.848806
9,R08,9,45397.555556,56297.826842,21873.126613,19973.434200


## 12. Generate leaderboard submission files

This section creates a small set of meaningful submissions:

1. the best absolute validation candidate,
2. the best leaderboard-safe candidate,
3. a few diverse high-performing candidates.

For competition submissions, the model is refitted on **all labeled training data** using the same chosen preprocessing and target strategy. This usually gives better leaderboard performance than training only on the 70% internal training split.

However, keep the internal-test result from earlier as the honest presentation estimate.


In [12]:
# ===============================================================
# 12. Generate submissions from selected candidates
# ===============================================================
def fit_candidate_on_all_labeled_and_predict_kaggle(row: pd.Series):
    profile = get_profile_by_name(row["feature_profile"])
    target_variant = row["target_variant"]
    svr_params = parse_svr_params(row["svr_params"])

    # Refit on all labeled data for competition prediction.
    df_fit, y_fit_log, _ = get_training_target_variant(df_all.reset_index(drop=False).rename(columns={"index": "original_index"}), target_variant)
    X_fit_raw = df_fit.drop(columns=[TARGET], errors="ignore")
    X_kaggle_raw = df_kaggle.copy()

    preproc_kwargs = {k: v for k, v in profile.items() if k != "name"}
    preprocessor = SVRRBFPreprocessor(**preproc_kwargs)
    X_fit = preprocessor.fit_transform(X_fit_raw, y_fit_log, scaled=True)
    X_kaggle = preprocessor.transform(X_kaggle_raw, scaled=True)

    model = make_model(svr_params)
    model.fit(X_fit, y_fit_log)
    pred_fit_log = np.asarray(model.predict(X_fit), dtype=float)
    pred_kaggle_log = np.asarray(model.predict(X_kaggle), dtype=float)

    correction = row["correction"]
    if correction == "none":
        factor = 1.0
    elif correction == "train_smearing":
        factor = float(np.mean(np.exp(np.asarray(y_fit_log) - pred_fit_log)))
    elif correction.startswith("fixed_"):
        factor = float(row["factor"])
    elif correction == "validation_multiplier":
        # This factor was learned from the internal validation set. Useful to test, but less stable.
        factor = float(row["factor"])
    else:
        factor = float(row["factor"])

    pred_kaggle_usd = safe_exp_to_usd(pred_kaggle_log, factor=factor)
    return pred_kaggle_usd, factor, model, preprocessor

# Choose candidate rows.
best_absolute = validation_results.sort_values("val_rmse").head(1)
best_safe = validation_results[validation_results["leaderboard_safe"]].sort_values("val_rmse").head(1)
# Include diverse candidates to learn from public leaderboard.
diverse = validation_results.sort_values("val_rmse").groupby(["feature_profile", "target_variant", "correction"], as_index=False).head(1).head(8)
submission_candidates = pd.concat([best_absolute, best_safe, diverse], ignore_index=True).drop_duplicates(
    subset=["feature_profile", "target_variant", "svr_params", "correction", "factor"]
).head(8)

submission_manifest = []
for i, (_, row) in enumerate(submission_candidates.iterrows(), start=1):
    pred_usd, factor_used, model, preprocessor = fit_candidate_on_all_labeled_and_predict_kaggle(row)
    sub = pd.DataFrame({
        "id": df_kaggle[ID_COL].astype(int),
        "annual.pay.usd": pred_usd.astype(float),
    })
    assert sub["id"].is_unique
    assert sub["annual.pay.usd"].notna().all()
    assert (sub["annual.pay.usd"] > 0).all()

    filename = f"submission_svr_rbf_{i:02d}_{row['feature_profile']}_{row['target_variant']}_{row['correction']}.csv"
    # Make file name safe.
    filename = re.sub(r"[^A-Za-z0-9_.-]+", "_", filename)
    sub_path = OUTPUT_DIR / filename
    sub.to_csv(sub_path, index=False)

    manifest_row = row.to_dict()
    manifest_row.update({
        "submission_file": str(sub_path),
        "factor_used_final_refit": factor_used,
        "prediction_mean": float(sub["annual.pay.usd"].mean()),
        "prediction_median": float(sub["annual.pay.usd"].median()),
        "prediction_min": float(sub["annual.pay.usd"].min()),
        "prediction_max": float(sub["annual.pay.usd"].max()),
    })
    submission_manifest.append(manifest_row)

submission_manifest_df = pd.DataFrame(submission_manifest)
submission_manifest_df.to_csv(OUTPUT_DIR / "submission_manifest.csv", index=False)

print("Generated submission files:")
display(submission_manifest_df[[
    "submission_file", "feature_profile", "target_variant", "C", "gamma", "epsilon", "correction", "factor_used_final_refit", "val_rmse", "prediction_mean", "prediction_median", "prediction_max", "leaderboard_safe"
]])


Generated submission files:


,submission_file,feature_profile,target_variant,C,gamma,epsilon,correction,factor_used_final_refit,val_rmse,prediction_mean,prediction_median,prediction_max,leaderboard_safe
0,svr_rbf_focused_outputs\submission_svr_rbf_01_...,top5_te,winsorized,0.5,scale,0.1,validation_multiplier,1.085,36970.354096,46016.527335,42839.898910,176104.594726,False
1,svr_rbf_focused_outputs\submission_svr_rbf_02_...,top5_te,winsorized,0.5,scale,0.1,fixed_1.08,1.080,36971.084166,45804.469605,42642.480021,175293.052815,True
2,svr_rbf_focused_outputs\submission_svr_rbf_03_...,top5_te,raw,0.5,scale,0.1,validation_multiplier,1.085,36971.828119,46042.080284,42753.575280,176324.631780,False
3,svr_rbf_focused_outputs\submission_svr_rbf_04_...,top5_te,raw,0.5,scale,0.1,fixed_1.08,1.080,36972.462182,45829.904799,42556.554196,175512.075873,True
4,svr_rbf_focused_outputs\submission_svr_rbf_05_...,top5_te,floor500,0.5,scale,0.1,validation_multiplier,1.085,36973.314339,46041.974876,42775.116447,176327.839537,False
5,svr_rbf_focused_outputs\submission_svr_rbf_06_...,top5_te,floor500,0.5,scale,0.1,fixed_1.08,1.080,36973.932582,45829.799876,42577.996095,175515.268848,True
6,svr_rbf_focused_outputs\submission_svr_rbf_07_...,top5_te,winsorized,0.5,scale,0.1,fixed_1.10,1.100,36977.651457,46652.700524,43432.155577,178539.220459,True
7,svr_rbf_focused_outputs\submission_svr_rbf_08_...,top5_te,raw,0.5,scale,0.1,fixed_1.10,1.100,36979.418027,46678.606740,43344.638533,178762.299501,True


## 13. What to do with the public leaderboard results

After submitting 2–3 files, manually record the public leaderboard score in `submission_manifest.csv` or a separate spreadsheet.

Recommended order of submissions:

1. best leaderboard-safe candidate,
2. best absolute validation candidate,
3. one diverse candidate with a different feature profile or target variant.

Do not overfit to the public leaderboard. The public score uses only about 30% of the test data; the private score uses the remaining 70%.


In [13]:
# ===============================================================
# 13. Final summary object
# ===============================================================
summary = {
    "notebook": "04_svr_rbf_focused_feature_engineering_and_modeling.ipynb",
    "algorithm": "SVR with RBF kernel only",
    "fast_mode": FAST_MODE,
    "split_summary": split_summary.to_dict(orient="records"),
    "n_validation_rows": int(len(validation_results)) if 'validation_results' in globals() else None,
    "best_validation_candidate": validation_results.iloc[0].to_dict() if 'validation_results' in globals() and len(validation_results) else None,
    "best_leaderboard_safe_candidate": validation_results[validation_results["leaderboard_safe"]].sort_values("val_rmse").iloc[0].to_dict() if 'validation_results' in globals() and len(validation_results[validation_results["leaderboard_safe"]]) else None,
    "important_warning": "Validation multiplier can overfit the validation split. Compare leaderboard-safe submissions as well.",
}
with open(OUTPUT_DIR / "focused_svr_rbf_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("Focused SVR RBF summary saved to:", (OUTPUT_DIR / "focused_svr_rbf_summary.json").resolve())
print("Main outputs are in:", OUTPUT_DIR.resolve())


Focused SVR RBF summary saved to: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_rbf_focused_outputs\focused_svr_rbf_summary.json
Main outputs are in: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\svr_rbf_focused_outputs
